## Preparation

TODO
- Install required packages
```python
pip install -r route_choice/requirements.txt
```

In [2]:
import numpy as np
import pandas as pd
import os

import re
import ast

from abc_rc import RouteChoiceModel
from model import RL
from definition import BLENetwork, LinkTransition, BLEIO
from algorithm import EMBLERouteChoice, BLEHybridMapmatching, dial_assignment, get_prev_link, get_shared_link, get_heading

def read_csv(file: str) -> pd.DataFrame:
    return pd.read_csv(file, engine="pyarrow")

## EM Algorithm for BLE Route Choice Model

TODO
- Change path to link, node, BLE signal data ,BLE logger location data and output directory
- Change model definition
- Change traning strategy (EMBLERouteChoice)

In [2]:
LINK_FILE = "../data/input/ble/link_post.csv"
NODE_FILE = "../data/input/ble/node_post.csv"
BLE_SIGNAL_FILE = "../data/input/ble/20230129_0845_20sec_off.csv"
BLE_LOGGER_FILE = "../data/input/ble/ble_nw.csv"
OUTPUT_DIR = "../data/output/ble/"

# Read data
df_link = read_csv(LINK_FILE)
df_node = read_csv(NODE_FILE)


In [3]:
cols = list(set(df_link.columns.tolist()) - {"linkid", "o", "d"})
# Fill missing values
df_link[cols] = df_link[cols].fillna(0)

# Check correlation
print(df_link[cols].corr())

# Drop ["congestion", "from_o"] in this demo
df_link = df_link.drop(columns=["congestion", "from_o"])

               width  platform     stair  congestion       esc    length  \
width       1.000000  0.638200 -0.650422         NaN -0.295725  0.054058   
platform    0.638200  1.000000 -0.570088         NaN -0.296374  0.096088   
stair      -0.650422 -0.570088  1.000000         NaN  0.519875  0.134840   
congestion       NaN       NaN       NaN         NaN       NaN       NaN   
esc        -0.295725 -0.296374  0.519875         NaN  1.000000  0.070100   
length      0.054058  0.096088  0.134840         NaN  0.070100  1.000000   
gate        0.076740 -0.384353 -0.539360         NaN -0.280400 -0.250000   
from_o           NaN       NaN       NaN         NaN       NaN       NaN   

                gate  from_o  
width       0.076740     NaN  
platform   -0.384353     NaN  
stair      -0.539360     NaN  
congestion       NaN     NaN  
esc        -0.280400     NaN  
length     -0.250000     NaN  
gate        1.000000     NaN  
from_o           NaN     NaN  


In [4]:
# Create network
network = BLENetwork(df_node, df_link)

# Create BLE instance
ble = BLEIO.load_from_csv(BLE_SIGNAL_FILE, BLE_LOGGER_FILE)
ble.set_network(network)

# Create model instance
model = RL(network)

# Create map-matching instance
mapmatching = BLEHybridMapmatching()

[2025-11-22 21:39:33,170] WARNING:ble.py 352:Some link attributes are out of the expected range (-10, 10).


In [5]:
# Perform EM algorithm
em_algorithm = EMBLERouteChoice(network, model, mapmatching)

# If max_iterations is 1, it's the same as normal map-matching
#result_str = em_algorithm.fit(ble, max_iterations=1)
result_str = em_algorithm.fit(ble)

with open(os.path.join(OUTPUT_DIR, "em_result.txt"), "w") as f:
    f.write(result_str)

EM Iteration 1
  E-step start


Trips:   0%|          | 0/1700 [00:00<?, ?it/s]

Trips:  37%|███▋      | 628/1700 [04:19<07:22,  2.42it/s]


KeyboardInterrupt: 

## Traffic Assignment

TODO
- Change path to link, node, demand files and output directory
- Change parameter file path
- Change model definition

In [3]:
LINK_FILE = "../data/input/ble/link_post.csv"
NODE_FILE = "../data/input/ble/node_post.csv"
DEMAND_FILE = "../data/input/ble/demand_demo.csv"
PARAMETER_FILE = "../data/output/ble/em_result.txt"
OUTPUT_DIR = "../data/output/ble/"

# Read data
df_link = read_csv(LINK_FILE)
df_node = read_csv(NODE_FILE)
df_demand = read_csv(DEMAND_FILE)

df_link = df_link.drop(columns=["congestion", "from_o"])

network = BLENetwork(df_node, df_link)

[2025-11-27 17:54:17,218] WARNING:ble.py 352:Some link attributes are out of the expected range (-10, 10).


In [4]:
# Create model
model = RL(network)

# Read Parameter
with open(PARAMETER_FILE, encoding="utf-8") as f:
    text = f.read()

match = re.search(r"parameter\s*=\s*(\[[^\]]+\])", text)
if match:
    param_list = ast.literal_eval(match.group(1))  # list[float]
else:
    raise ValueError("Failed to extract parameters from input/result.txt")
params = np.array(param_list, dtype=np.float32)

In [5]:
# Prepare network properties
prev_link = get_prev_link(network)
shared_link = get_shared_link(network)
heading = get_heading(network)

# Assignment for each OD pair
o_node_ids = df_demand["OriginNodeID"].to_numpy()
d_node_ids = df_demand["DestinationNodeID"].to_numpy()
demands = df_demand["Demand"].to_numpy()

link_flow = np.zeros(network.n_link, dtype=np.float32)
for i in range(len(df_demand)):
    _, path = network.get_shortest_path(o_node_ids[i], d_node_ids[i])
    if len(path) > 1:
        o_link_id = path[0]
        d_link_id = path[-1]

        link_transition = LinkTransition(i, o_link_id, None, d_node_ids[i], [], model)
        
        P2 = model.calculate_transition_probabilities(link_transition, params)
        u_od = np.zeros((network.n_link, network.n_link), dtype=np.float32)
        u_od[network.link_id2idx[o_link_id], network.link_id2idx[d_link_id]] = demands[i]

        link_flow_tmp = dial_assignment(
            u_od,
            network.link_adj_matrix.toarray().astype(np.float32),
            network.link_dist_matrix.astype(np.float32),
            prev_link,
            P_2=P2,
            shared_links=shared_link,
            headings=heading,
        )
        link_flow += link_flow_tmp

# Output result
df_flow = pd.DataFrame({
    "LinkID": network.link_list,
    "LinkFlow": link_flow,
})
output_file = os.path.join(OUTPUT_DIR, "link_flow.csv")
df_flow.to_csv(output_file, index=False)

In [7]:
df_flow.sum()

LinkID      1035.0
LinkFlow      20.0
dtype: float64